# EDA dataset Fresh and Stale Classification

Phân tích cấu trúc, chất lượng, kích thước, màu sắc, độ sáng, ảnh trùng và trực quan hóa dữ liệu.

**Dataset Kaggle:** `/kaggle/input/datasets/swoyam2609/fresh-and-stale-classification/dataset`

In [ ]:
from pathlib import Path
import hashlib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
from IPython.display import display

warnings.filterwarnings('ignore')
ImageFile.LOAD_TRUNCATED_IMAGES = False
sns.set_theme(style='whitegrid', context='notebook')
DATASET_DIR = Path('/kaggle/input/datasets/swoyam2609/fresh-and-stale-classification/dataset')
OUTPUT_DIR = Path('/kaggle/working/eda_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tif', '.tiff', '.webp'}
if not DATASET_DIR.exists():
    raise FileNotFoundError(f'Không tìm thấy dataset: {DATASET_DIR}')

# Quét file, đọc metadata và kiểm tra ảnh hỏng.
image_paths = sorted(p for p in DATASET_DIR.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
if not image_paths:
    raise RuntimeError('Không tìm thấy file ảnh trong dataset.')
def class_of(path):
    parts = path.relative_to(DATASET_DIR).parts
    return parts[0] if len(parts) > 1 else 'unknown'
records = []
for path in image_paths:
    row = {'path': str(path), 'relative_path': str(path.relative_to(DATASET_DIR)), 'filename': path.name,
           'class': class_of(path), 'extension': path.suffix.lower(), 'file_size_kb': path.stat().st_size / 1024,
           'width': np.nan, 'height': np.nan, 'channels': np.nan, 'mode': None, 'format': None,
           'is_valid': False, 'error': None, 'md5': None}
    try:
        with Image.open(path) as im: im.verify()
        with Image.open(path) as im:
            row.update(width=im.width, height=im.height, mode=im.mode, format=im.format,
                       channels=len(im.getbands()), is_valid=True)
        row['md5'] = hashlib.md5(path.read_bytes()).hexdigest()
    except Exception as exc:
        row['error'] = f'{type(exc).__name__}: {exc}'
    records.append(row)
metadata = pd.DataFrame(records)
metadata['aspect_ratio'] = metadata['width'] / metadata['height']
metadata['megapixels'] = metadata['width'] * metadata['height'] / 1e6
valid = metadata[metadata.is_valid].copy()
metadata.to_csv(OUTPUT_DIR / 'image_metadata.csv', index=False)
print(f'Tổng file: {len(metadata):,} | Hợp lệ: {len(valid):,} | Lỗi: {len(metadata)-len(valid):,}')
display(metadata.head())
if (~metadata.is_valid).any(): display(metadata.loc[~metadata.is_valid, ['relative_path', 'error']])

# Số lượng và cân bằng lớp.
class_counts = metadata.groupby('class').size().sort_values(ascending=False).rename('count').to_frame()
class_counts['percentage'] = class_counts['count'] / len(metadata) * 100
class_counts['relative_to_largest'] = class_counts['count'] / class_counts['count'].max()
display(class_counts.style.format({'percentage': '{:.2f}%', 'relative_to_largest': '{:.3f}'}))
print(f'Số lớp: {len(class_counts)} | Tỷ lệ lớn nhất/nhỏ nhất: {class_counts.iloc[0,0]/class_counts.iloc[-1,0]:.2f}')
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=class_counts.reset_index(), x='class', y='count', hue='class', legend=False, palette='Set2', ax=axes[0])
axes[0].set_title('Số lượng ảnh theo lớp'); axes[0].tick_params(axis='x', rotation=30)
ext = metadata.extension.value_counts(); axes[1].pie(ext, labels=ext.index, autopct='%.1f%%', startangle=90)
axes[1].set_title('Tỷ lệ định dạng ảnh'); plt.tight_layout(); plt.show()
class_counts.to_csv(OUTPUT_DIR / 'class_counts.csv')

# Kích thước, mode, kênh màu, dung lượng và ảnh trùng.
display(valid[['width','height','aspect_ratio','megapixels','file_size_kb']].describe().T)
display(valid[['channels','mode','format']].melt().groupby(['variable','value']).size().rename('count').to_frame())
summary = valid.groupby('class')[['width','height','aspect_ratio','megapixels','file_size_kb']].agg(['min','median','mean','max'])
display(summary.round(3)); summary.to_csv(OUTPUT_DIR / 'size_summary_by_class.csv')
duplicates = valid[valid.duplicated('md5', keep=False)].sort_values('md5')
print(f'Nhóm ảnh trùng: {duplicates.md5.nunique()} | Số file trùng: {len(duplicates)}')
display(duplicates[['class','relative_path','md5']].head(30)); duplicates.to_csv(OUTPUT_DIR / 'duplicate_images.csv', index=False)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, col, title in zip(axes.ravel(), ['width','height','aspect_ratio','file_size_kb'], ['Chiều rộng','Chiều cao','Tỷ lệ khung hình','Dung lượng (KB)']):
    sns.histplot(data=valid, x=col, hue='class', bins=30, element='step', stat='density', common_norm=False, ax=ax)
    ax.set_title(f'Phân bố {title}')
plt.tight_layout(); plt.show()

# Đặc trưng pixel trên ảnh resize 128x128.
def pixel_features(path):
    with Image.open(path) as im: arr = np.asarray(im.convert('RGB').resize((128, 128)), dtype=np.float32) / 255
    gray = arr.mean(axis=2)
    return {'brightness_mean': gray.mean(), 'brightness_std': gray.std(), 'red_mean': arr[:,:,0].mean(),
            'green_mean': arr[:,:,1].mean(), 'blue_mean': arr[:,:,2].mean(),
            'saturation_mean': (arr.max(axis=2)-arr.min(axis=2)).mean(),
            'dark_pixel_fraction': (gray < .2).mean(), 'bright_pixel_fraction': (gray > .8).mean()}
features = pd.DataFrame([pixel_features(p) for p in valid.path], index=valid.index)
valid = pd.concat([valid, features], axis=1)
valid.to_csv(OUTPUT_DIR / 'valid_images_with_pixel_features.csv', index=False)
pixel_summary = valid.groupby('class')[list(features.columns)].agg(['mean','std','min','max'])
display(pixel_summary.round(4)); pixel_summary.to_csv(OUTPUT_DIR / 'pixel_feature_summary_by_class.csv')
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for name, group in valid.groupby('class'):
    axes[0,0].hist(group.brightness_mean, bins=25, alpha=.55, label=name)
    axes[0,1].hist(group.brightness_std, bins=25, alpha=.55, label=name)
    axes[1,0].hist(group.saturation_mean, bins=25, alpha=.55, label=name)
    axes[1,1].scatter(group.brightness_mean, group.saturation_mean, s=18, alpha=.45, label=name)
for ax, title in zip(axes.ravel(), ['Độ sáng trung bình','Độ tương phản','Độ bão hòa','Độ sáng và bão hòa']): ax.set_title(title); ax.legend(title='class')
plt.tight_layout(); plt.show()

# Hiển thị tối đa 12 ảnh mẫu mỗi lớp.
rng = np.random.default_rng(231)
for name in sorted(valid['class'].unique()):
    group = valid[valid['class'] == name].iloc[rng.choice(sum(valid['class'] == name), min(12, sum(valid['class'] == name)), replace=False)]
    fig, axes = plt.subplots(3, 4, figsize=(14, 10)); axes = axes.ravel()
    for ax, (_, row) in zip(axes, group.iterrows()):
        with Image.open(row.path) as im: ax.imshow(im.convert('RGB'))
        ax.set_title(f'{row.filename}\n{int(row.width)}x{int(row.height)}', fontsize=8); ax.axis('off')
    for ax in axes[len(group):]: ax.axis('off')
    fig.suptitle(f'Ảnh mẫu - {name}'); plt.tight_layout(); plt.show()
print(f'Đã lưu CSV và biểu đồ tại: {OUTPUT_DIR}')

## Gợi ý kết luận EDA

- Nếu tỷ lệ lớp mất cân bằng cao, dùng stratified split và cân nhắc class weight/oversampling.
- Nếu kích thước ảnh không đồng nhất, cần thống nhất kích thước khi huấn luyện.
- Loại ảnh hỏng; kiểm tra ảnh trùng để tránh rò rỉ giữa train và validation/test.
- So sánh độ sáng, độ bão hòa và RGB giữa các lớp để phát hiện tín hiệu hoặc bias do điều kiện chụp.
- Kết quả được lưu trong `/kaggle/working/eda_outputs/`.